# Risø Conference Proceedings 2026 - lab source - LM optimizer
This notebook contains the accompanying processing code for the grazing-indidence case to support the Risø Conference Proceedings 2026 paper:

Ball, J. A. D., Andreasen, J. W., Angelis, S. D., Wright, J. P., & Detlefs, C. (2026, July 9). Multi-Beam 3DXRD. IOP Conference Series: Materials Science and Engineering. 46th Risø International Symposium on Materials Science: Characterization of evolving microstructures in metals, DTU Risø Campus, Roskilde, Denmark. Accepted for publication.

This notebook is very similar to the general lab case, but we use a Levenberg–Marquardt approach to optimize much faster.

In [ ]:
import os
# don't hog GPU memory (if you have one)
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "0"

import jax
jax.config.update("jax_enable_x64", True)  # required for accuracy, especially strains
import jax.numpy as jnp

import time
from functools import partial

import scipy
from scipy.spatial.transform import Rotation
from matplotlib import pyplot as plt

import ImageD11.grain
import ImageD11.unitcell
import ImageD11.indexing
from xfab.symmetry import ROTATIONS, Umis

from ImageD11.nbGui.nb_utils import plot_grain_positions, plot_all_ipfs

import anri.crystal, anri.diffract, anri.geom, anri.fwd

start = time.time()

## Constants

In [ ]:
### BEAM

# Energies in eV
# https://xdb.lbl.gov/Section1/Table_1-2.pdf
k_alpha_1_eV = 24209.7 
k_alpha_2_eV = 24002.0

def eV_to_wavelength_A(energy_eV):
    # L = hc/E
    # Must convert eV to Joules first
    wavelength_A = (scipy.constants.h * scipy.constants.c)/(scipy.constants.e*energy_eV)*1e10
    return wavelength_A

k_alpha_1_A = eV_to_wavelength_A(k_alpha_1_eV)
k_alpha_2_A = eV_to_wavelength_A(k_alpha_2_eV)

k_alpha_1_A, k_alpha_2_A

## Crystallography
We'll start with HCP Ti

In [ ]:
struc = anri.crystal.Structure.from_cif("../../../tests/data/cif/Ti.cif")
struc.lattice_parameters

We generate some HKLs. This needs a single wavelength supplied to compute the $2\theta$ angles for the reflections table, but it won't be directly used here.

In [ ]:
struc.make_hkls(dsmax=1.0, wavelength=k_alpha_1_A)
struc.rings_table

## Phantom sample

Let's define our phantom sample.  
We define everything in the sample coordinate system (on top of the goniometer).  
We'll randomly generate some grains within a box.

In [ ]:
rng = 28  # chosen by fair dice roll, guaranteed to be random
key = jax.random.key(rng)

n_grains = 450

### Positions
sample_width = 1000.0
translations_sample = jax.random.uniform(key, shape=(n_grains,3,), minval=-sample_width/2, maxval=sample_width/2)

### Orientations
U_matrices = Rotation.random(n_grains, rng=rng).as_matrix()
UB_matrices = U_matrices @ struc.B
UBI_matrices = jnp.linalg.inv(UB_matrices)

### Volumes - just for visualisation purposes!
radii_sigma = 0.2
radii_mean = 100.0
radii = jax.random.lognormal(key, shape=(n_grains,), sigma=radii_sigma) * radii_mean
volumes = (4./3)*jnp.pi*(radii**3)

We can use ImageD11 to plot this phantom.

In [ ]:
ref_unitcell = ImageD11.unitcell.unitcell(struc.lattice_parameters, symmetry=struc.sgno)
grains = [ImageD11.grain.grain(UBI_matrices[i], translation=translations_sample[i]) for i in range(n_grains)]

for i, g in enumerate(grains):
    g.ref_unitcell = ref_unitcell
    g.intensity_info = f"mean = {volumes[i]}"

plot_grain_positions(grains, 'z', size_scaling=0.1)
plot_all_ipfs(grains)

## Forward projection
Now we can generate our scattering vectors and translate them into the lab frame, then into the detector.  
This will yield centroids, which are arrays of `[slow, fast, omega]` which represent the centre-of-mass positions of the peaks on the detector surface.  
We must define our goniometer and detector positions:

In [ ]:
### Goniometer

wedge = 0.0
chi = 0.0
y0 = 0.0

### Detector
y_center = 2048.0
z_center = 2048.0
y_size = 100.0
z_size = 100.0
tilt_x = 0.0
tilt_y = 0.0
tilt_z = 0.0
distance = 515e3
o11 = 1
o12 = 0
o21 = 0
o22 = 1

# not official parameters but useful for plotting later
det_size_s = 4096
det_size_f = 4096

# Get change-of-basis parameters to go from lab to detector space:
det_trans, beam_cen_shift, x_distance_shift = anri.geom.detector_transforms(
    y_center,
    y_size,
    tilt_y,
    z_center,
    z_size,
    tilt_z,
    tilt_x,
    distance,
    o11,
    o12,
    o21,
    o22
)

# Get detector unit vectors (slow, fast directions) in lab frame:
sc_lab, fc_lab, norm_lab = anri.geom.detector_basis_vectors_lab(det_trans, beam_cen_shift, x_distance_shift)

In [ ]:
# we get plus/minus Friedel pairs
# we also get boolean masks for validity - sometimes no solution exists for the Ewald condition, so we never see the peak.

# unit beam wavevector, lab frame
k_in_lab_hat = jnp.array([1., 0, 0,])

centroid_a1_p, valid_a1_p = anri.fwd.get_centroid_box_all(UBI_matrices, translations_sample,
                                              struc.ringhkls_arr,
                                              1.0, k_alpha_1_A, k_in_lab_hat, 0, 0,
                                              wedge, chi,
                                              sc_lab, fc_lab, norm_lab)
centroid_a1_m, valid_a1_m = anri.fwd.get_centroid_box_all(UBI_matrices, translations_sample,
                                              struc.ringhkls_arr,
                                              -1.0, k_alpha_1_A, k_in_lab_hat, 0, 0,
                                              wedge, chi,
                                              sc_lab, fc_lab, norm_lab)

centroid_a2_p, valid_a2_p = anri.fwd.get_centroid_box_all(UBI_matrices, translations_sample,
                                              struc.ringhkls_arr,
                                              1.0, k_alpha_2_A, k_in_lab_hat, 0, 0,
                                              wedge, chi,
                                              sc_lab, fc_lab, norm_lab)
centroid_a2_m, valid_a2_m = anri.fwd.get_centroid_box_all(UBI_matrices, translations_sample,
                                              struc.ringhkls_arr,
                                              -1.0, k_alpha_2_A, k_in_lab_hat, 0, 0,
                                              wedge, chi,
                                              sc_lab, fc_lab, norm_lab)

# join Friedel pairs into single datasets:
centroid_a1 = jnp.concatenate([centroid_a1_p[valid_a1_p], centroid_a1_m[valid_a1_m]])
centroid_a2 = jnp.concatenate([centroid_a2_p[valid_a2_p], centroid_a2_m[valid_a2_m]])

# flatten into (N, 3)
centroid_a1 = centroid_a1.reshape(-1, 3)
centroid_a2 = centroid_a2.reshape(-1, 3)

# mask to detector pixel range
m_a1 = (centroid_a1[:, 0] > 0) & (centroid_a1[:, 0] < det_size_s) & (centroid_a1[:, 1] > 0) & (centroid_a1[:, 1] < det_size_f)
m_a2 = (centroid_a2[:, 0] > 0) & (centroid_a2[:, 0] < det_size_s) & (centroid_a2[:, 1] > 0) & (centroid_a2[:, 1] < det_size_f)
centroid_a1 = centroid_a1[m_a1]
centroid_a2 = centroid_a2[m_a2]

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12,20), constrained_layout=True)
axs[0].scatter(centroid_a1[:, 1], centroid_a1[:, 0], label=r'$K_{\alpha_{1}}$ peaks', s=2)
axs[0].scatter(centroid_a2[:, 1], centroid_a2[:, 0], label=r'$K_{\alpha_{2}}$ peaks', s=2)
axs[0].set_aspect(1)
axs[0].set(xlabel='Detector fast', ylabel='Detector slow', title='Whole detector view')
axs[0].legend(loc='upper right')
axs[1].scatter(centroid_a1[:, 1], centroid_a1[:, 0], label=r'$K_{\alpha_{1}}$ peaks', s=2)
axs[1].scatter(centroid_a2[:, 1], centroid_a2[:, 0], label=r'$K_{\alpha_{2}}$ peaks', s=2)
axs[1].set_aspect(1)
axs[1].set(xlabel='Detector fast', ylabel='Detector slow', xlim=(890, 940), ylim=(890, 940), title='Detail view')
axs[1].legend(loc='upper right')
plt.show()

## Scattering vector identification
With the peaks forward-projected onto the detector, we now 'forget' which wavelength each peak came from.  
We compute scattering vectors in the sample frame for *all* peaks, twice, assuming each wavelength.  
We also forget the origin of diffraction of each centroid.

In [ ]:
centroid = jnp.concatenate([centroid_a1, centroid_a2])

## Simulating experimental error
We now make a sensible effort to "spoil" the measured centroids to account for experimental errors.  
This can be done more accurately (and will be in the future when we simulate intensity profiles on the detector) but to first order this should be a reasonable approach.

In [ ]:
def spoil_centroids(centroids, width_px, width_omega):
    npks = centroids.shape[0]
    px_error_vector = jax.random.uniform(key, shape=(npks,2), minval=-width_px/2, maxval=width_px/2)
    omega_error_vector = jax.random.uniform(key, shape=(npks,), minval=-width_omega/2, maxval=width_omega/2)
    error_vector = jnp.column_stack((px_error_vector, omega_error_vector))
    centroids = centroids + error_vector
    return centroids

ostep = 0.1  # realistic omega step
centroid_with_error = spoil_centroids(centroid, 1.0, ostep)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12,20), constrained_layout=True)
axs[0].scatter(centroid[:, 1], centroid[:, 0], label=r'Peaks without error')
axs[0].scatter(centroid_with_error[:, 1], centroid_with_error[:, 0], s=10, label=r'Peaks with error')
axs[0].set_aspect(1)
axs[0].set(xlabel='Detector fast', ylabel='Detector slow', title='Whole detector view')
axs[0].legend(loc='upper right')
axs[1].scatter(centroid[:, 1], centroid[:, 0], label=r'Peaks without error')
axs[1].scatter(centroid_with_error[:, 1], centroid_with_error[:, 0], s=10, label=r'Peaks with error')
axs[1].set_aspect(1)
axs[1].set(xlabel='Detector fast', ylabel='Detector slow', xlim=(890, 940), ylim=(890, 940), title='Detail view')
axs[1].legend(loc='upper right')
plt.show()

## Backward projection
We can now project backwards from detector space to scattering vectors in sample space $\mathbf{g_s}$

In [ ]:
# All these primitive functions are written for single vectors
# We write the overall function for a single vector, then vmap it over many vectors for speed:

@jax.jit
def detector_to_q(slow, fast, omega, wavelength, k_in_lab_hat, origin_sample):
    # peak vector in lab frame
    peak_lab = anri.geom.det_to_lab(slow, fast, det_trans, beam_cen_shift, x_distance_shift)
    # rotate origin into sample frame
    origin_lab = anri.geom.sample_to_lab(origin_sample, omega, wedge, chi, 0.0, 0.0)
    
    # convert peak vector to k_out, subtracts off the origin
    k_out_lab_norm = anri.diffract.peak_lab_to_k_out(peak_lab, origin_lab, wavelength)
    # normalise k_in by wavelength
    k_in_lab_norm = anri.diffract.scale_norm_k(k_in_lab_hat, wavelength)
    # simply q = k_out - k_in
    q_lab = anri.diffract.k_to_q_lab(k_in_lab_norm, k_out_lab_norm)
    # rotate into sample frame
    q_sample = anri.geom.lab_to_sample(q_lab, omega, wedge, chi, 0.0, 0.0)
    return q_sample

# the vmap operation
detector_to_q_vec = jax.vmap(detector_to_q, in_axes=(0, 0, 0, 0, 0, None))

In [ ]:
# Compute scattering vectors for all peaks assuming each wavelength

kvecs = jnp.broadcast_to(k_in_lab_hat, centroid_with_error.shape)
wavelengths_a1 = jnp.full(centroid_with_error.shape[0], k_alpha_1_A)
wavelengths_a2 = jnp.full(centroid_with_error.shape[0], k_alpha_2_A)

q_sample_a1 = detector_to_q_vec(centroid_with_error[:, 0], centroid_with_error[:, 1], centroid_with_error[:, 2], wavelengths_a1, kvecs, jnp.array([0., 0., 0.]))
q_sample_a2 = detector_to_q_vec(centroid_with_error[:, 0], centroid_with_error[:, 1], centroid_with_error[:, 2], wavelengths_a2, kvecs, jnp.array([0., 0., 0.]))

## *k*-d tree search
We now construct a *k*-d tree in sample space.  
The idea is that half of the observed scattering vectors will be correctly computed with $\lambda = K_{\alpha_{1}}$, and the other half will be correctly computed with $\lambda = K_{\alpha_{2}}$.  
As these duplicated scattering vectors come from a single set of reciprocal lattice points, the two datasets should 'overlap' if and only if the wavelength was correctly chosen for a given scattering vector.
We look for 'overlapping' (i.e. duplicate) scattering vectors in sample space using a *k*-d tree.

In practice, we first do this via a sparse distance matrix with a large distance tolerance, to determine where the sensible cutoff should be:

In [ ]:
kd_a2 = scipy.spatial.cKDTree(q_sample_a2)

# Find the pairs
distances, indices = kd_a2.query(q_sample_a1, k=1, distance_upper_bound=0.03)
valid_mask = jnp.isfinite(distances)
valid_distances = distances[valid_mask]

fig, ax = plt.subplots()
ax.hist(valid_distances, bins=100)
ax.set(xlabel='Distance in g-vector space', ylabel='Count', title='Histogram of g-vector neighbour distances')
plt.show()

We can perhaps discern that a sensible cutoff would be `0.0015` to get the first spike

In [ ]:
# Find the pairs
distances, indices = kd_a2.query(q_sample_a1, k=1, distance_upper_bound=0.0015)
valid_mask = jnp.isfinite(distances)
valid_distances = distances[valid_mask]

fig, ax = plt.subplots()
ax.hist(valid_distances, bins=100)
ax.set(xlabel='Distance in g-vector space', ylabel='Count', title='Histogram of g-vector neighbour distances')
plt.show()

Now we need to average or combine our observations of the g-vectors.

We have three sources of g-vectors we can possibly index:

- G-vectors only from $K_{\alpha_{1}}$
- G-vectors only from $K_{\alpha_{2}}$
- Averaged/combined observations of both

In [ ]:
mask_a1 = valid_mask
mask_a2 = indices[mask_a1]

q_sample_a1_paired = q_sample_a1[mask_a1]
q_sample_a2_paired = q_sample_a2[mask_a2]

# concatenate
q_sample_cat = jnp.concatenate((q_sample_a1_paired, q_sample_a2_paired))

We can confirm that the deduplication succeeded by masking the centroids array in the same way:

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12,20), constrained_layout=True)
axs[0].scatter(centroid_with_error[:, 1], centroid_with_error[:, 0], s=50, label=r'$K_{\alpha_{1}}$ and $K_{\alpha_{1}}$ peaks')
axs[0].scatter(centroid_with_error[:, 1][mask_a1], centroid_with_error[:, 0][mask_a1], s=10, label=r'Deduplicated peaks')
axs[0].set_aspect(1)
axs[0].set(xlabel='Detector fast', ylabel='Detector slow', title='Whole detector view')
axs[0].legend(loc='upper right')

axs[1].scatter(centroid_with_error[:, 1], centroid_with_error[:, 0], s=50, label=r'$K_{\alpha_{1}}$ and $K_{\alpha_{1}}$ peaks')
axs[1].scatter(centroid_with_error[:, 1][mask_a1], centroid_with_error[:, 0][mask_a1], s=10, label=r'Deduplicated peaks')
axs[1].set_aspect(1)
axs[1].set(xlabel='Detector fast', ylabel='Detector slow', xlim=(890, 940), ylim=(890, 940), title='Detail view')
axs[1].legend(loc='upper right')
plt.show()

## Indexing the recovered scattering vectors
### Just $K_{\alpha_{1}}$

In [ ]:
# make an indexer
idx_a1 = ImageD11.indexing.indexer(unitcell=ref_unitcell, gv=q_sample_a1_paired, minpks=80)
idx_a1.ds_tol = 0.005
idx_a1.assigntorings()
idx_a1.hkl_tol = 0.02
idx_a1.cosine_tol = 0.002
idx_a1.score_all_pairs()
idx_a1.saveubis('grains_found_a1.ubi')

### Just $K_{\alpha_{2}}$

In [ ]:
# make an indexer
idx_a2 = ImageD11.indexing.indexer(unitcell=ref_unitcell, gv=q_sample_a2_paired, minpks=80)
idx_a2.ds_tol = 0.005
idx_a2.assigntorings()
idx_a2.hkl_tol = 0.02
idx_a2.cosine_tol = 0.002
idx_a2.score_all_pairs()
idx_a2.saveubis('grains_found_a2.ubi')

### Combined scattering vectors

In [ ]:
# make an indexer
idx_both = ImageD11.indexing.indexer(unitcell=ref_unitcell, gv=q_sample_cat, minpks=160)
idx_both.ds_tol = 0.005
idx_both.assigntorings()
idx_both.hkl_tol = 0.02
idx_both.cosine_tol = 0.002
idx_both.score_all_pairs()
idx_both.saveubis('grains_found_avg.ubi')

## Grain position and UBI refinement
We can now try to refine the positions and UBIs of the grains we've found.  
We do this in three ways:  
- Indexed with $K_{\alpha_{1}}$ g-vectors, refined with $K_{\alpha_{1}}$ centroids
- Indexed with $K_{\alpha_{2}}$ g-vectors, refined with $K_{\alpha_{2}}$ centroids
- Indexed with merged g-vectors, refined with both $K_{\alpha_{1}}$ and $K_{\alpha_{2}}$ centroids

We implement a gradient-aware minimiser using ADAM from the Optax library. The loss function itself is identical to `makemap.py` in ImageD11.

### Peak to grain assignment
We have to establish static peak to grain assignments before we can refine, so grains do not compete for peaks. We do this just like ImageD11 - each peak is assigned to the grain that best indexes it (yields $hkl$ values closest to integer).

In [ ]:
@jax.jit
def assign(ubi, q_sample):
    hklf = (ubi @ q_sample.T).T
    hkli = jnp.rint(hklf)
    hkle = jnp.linalg.norm(hklf - hkli)
    return hkle, hkli

assign_peaks = jax.vmap(assign, in_axes=[None, 0])
assign_grains = jax.vmap(assign_peaks, in_axes=[0, None])

# compute hkl errors
hkle_a1, hkli_a1     = assign_grains(jnp.array(idx_a1.ubis),   q_sample_a1[mask_a1])
hkle_a2, hkli_a2     = assign_grains(jnp.array(idx_a2.ubis),   q_sample_a2[mask_a2])
hkle_both, hkli_both = assign_grains(jnp.array(idx_both.ubis), q_sample_cat)

# get grain assignments (minimised error per peak)
ga_a1   = jnp.argmin(hkle_a1, axis=0)
ga_a2   = jnp.argmin(hkle_a2, axis=0)
ga_both = jnp.argmin(hkle_both, axis=0)

# get hkli per peak
hkli_a1   = jnp.squeeze(jnp.take_along_axis(hkli_a1,   ga_a1[None, :, None], axis=0))
hkli_a2   = jnp.squeeze(jnp.take_along_axis(hkli_a2,   ga_a2[None, :, None], axis=0))
hkli_both = jnp.squeeze(jnp.take_along_axis(hkli_both, ga_both[None, :, None], axis=0))

# max grains per peak
M = max(jnp.unique(ga_a1, return_counts=True)[1].max(), jnp.unique(ga_a2, return_counts=True)[1].max(), jnp.unique(ga_both, return_counts=True)[1].max())
M

### Refinement code

In [ ]:
@jax.jit
def solve_ub_analytical(hkl_int, q_sample, mask):
    """UB = (Qs^T H) (H^T H)^-1, with masked rows zeroed out."""
    m = mask.astype(q_sample.dtype)[:, None]

    H  = hkl_int  * m           # (N, 3)
    Qs = q_sample * m           # (N, 3)

    HHT  = H.T @ H              # (3, 3), symmetric
    QsHT = Qs.T @ H             # (3, 3)

    # ridge: grains with <3 usable peaks (or all-padding rows from top_k)
    # would otherwise return NaN and poison the whole vmap batch
    eps = 1e-12 * jnp.trace(HHT) + 1e-15
    HHT = HHT + eps * jnp.eye(3, dtype=HHT.dtype)

    # X = QsHT @ inv(HHT); HHT symmetric => X = solve(HHT, QsHT.T).T
    return jnp.linalg.solve(HHT, QsHT.T).T


@jax.jit
def per_peak_err(ubi, origin, cen, hkl, wave, kvec, mask):
    """|Δhkl| for every peak in a grain's buffer. Masked rows -> inf."""
    q = detector_to_q_vec(cen[:, 0], cen[:, 1], cen[:, 2], wave, kvec, origin)
    d = (ubi @ q.T).T - hkl
    e = jnp.sqrt(jnp.sum(d * d, axis=1) + 1e-24)
    return jnp.where(mask, e, jnp.inf)

per_peak_err_vmap = jax.vmap(per_peak_err, in_axes=(0, 0, 0, 0, 0, 0, 0))

@partial(jax.jit, static_argnums=(7,))
def refine_grain(
    initial_ubi,          # kept for call-site compatibility; unused (as before)
    initial_origin,
    cen_obs,
    hkl_int,
    wavelength,
    kvecs,
    mask,
    num_steps=20,
    init_damping=1e-2,    # was learning_rate; same positional slot
    scaling_factor=1000.0,
):
    mf  = mask.astype(cen_obs.dtype)
    npk = jnp.sum(mf) + 1e-10

    def residuals(x):
        """Mask-weighted hkl residual, flattened, for a trial origin."""
        origin = x * scaling_factor
        q_sample = detector_to_q_vec(
            cen_obs[:, 0], cen_obs[:, 1], cen_obs[:, 2],
            wavelength, kvecs, origin,
        )
        ub   = solve_ub_analytical(hkl_int, q_sample, mask)
        hklf = jnp.linalg.solve(ub, q_sample.T).T          # (N, 3)
        return ((hklf - hkl_int) * mf[:, None]).ravel()    # (3N,)

    def mean_hkl_error(r):
        d = r.reshape(-1, 3)
        return jnp.sum(jnp.sqrt(jnp.sum(d * d, axis=1) + 1e-24)) / npk

    jac = jax.jacfwd(residuals)      # (3N, 3) — only 3 JVPs

    def lm_step(carry, _):
        x, lam = carry

        r = residuals(x)
        J = jac(x)

        JTJ = J.T @ J
        g   = J.T @ r
        I3  = jnp.eye(3, dtype=JTJ.dtype)

        damp = lam * (jnp.diag(jnp.diag(JTJ)) + 1e-12 * I3)
        dx   = jnp.linalg.solve(JTJ + damp, -g)

        x_try = x + dx
        r_try = residuals(x_try)

        # NaN in r_try makes this False, so a bad step is rejected automatically
        improved = jnp.sum(r_try * r_try) < jnp.sum(r * r)
        x_next   = jnp.where(improved, x_try, x)
        lam_next = jnp.clip(jnp.where(improved, lam / 3.0, lam * 5.0), 1e-10, 1e10)

        return (x_next, lam_next), mean_hkl_error(r)

    x0 = initial_origin / scaling_factor
    lam0 = jnp.asarray(init_damping, x0.dtype)

    (x_final, _), loss_history = jax.lax.scan(
        lm_step, (x0, lam0), None, length=num_steps
    )

    final_origin = x_final * scaling_factor
    q_sample = detector_to_q_vec(
        cen_obs[:, 0], cen_obs[:, 1], cen_obs[:, 2],
        wavelength, kvecs, final_origin,
    )
    final_ubi = jnp.linalg.inv(solve_ub_analytical(hkl_int, q_sample, mask))

    final_loss = mean_hkl_error(residuals(x_final))
    loss_history = jnp.concatenate([loss_history, final_loss[None]])

    return final_ubi, final_origin, loss_history

refine_vmap = jax.vmap(
    refine_grain, 
    in_axes=(0, 0, 0, 0, 0, 0, 0, None, None, None)
)

### Just $K_{\alpha_{1}}$

In [ ]:
%%time

all_origins_a1 = jnp.zeros((len(idx_a1.ubis), 3))

def get_compressed_grain_data(grain_id, assignments, cen, hkl, wave, kvec):
    # Create a boolean mask for this specific grain
    grain_mask = (assignments == grain_id)
    
    # Use top_k to find the indices of the True values.
    # This identifies exactly which rows in the filtered arrays belong to this grain.
    # Since we want static shapes, we always take M indices.
    _, indices = jax.lax.top_k(grain_mask.astype(jnp.int32), M)
    
    # Gather the data for this grain
    # If the grain has < M peaks, top_k pads with the lowest-index zero
    # entries (i.e. other grains' peaks); 'mask' is correctly False there.
    return {
        "cen": cen[indices],
        "hkl": hkl[indices],
        "wave": wave[indices],
        "kvec": kvec[indices],
        "mask": grain_mask[indices]
    }

# Vectorize the packing over all grain IDs
v_pack = jax.vmap(
    get_compressed_grain_data, 
    in_axes=(0, None, None, None, None, None)
)

# Pack the data into grain-specific buffers (num_grains, M, ...)
grain_ids_a1 = jnp.arange(len(idx_a1.ubis))
compressed_a1 = v_pack(grain_ids_a1, ga_a1, centroid_with_error[mask_a1], hkli_a1, wavelengths_a1[mask_a1], kvecs[mask_a1])

mask_r = compressed_a1["mask"]

for it in range(3):
    ubis, origins, losses = refine_vmap(
        jnp.array(idx_a1.ubis), all_origins_a1,
        compressed_a1["cen"],  compressed_a1["hkl"], compressed_a1["wave"],
        compressed_a1["kvec"], mask_r,
        5, 1e-2, 1000.0)

    # outlier rejection - unassign peaks with more than 4x the median hkl error
    # LM is very sensitive!
    e   = per_peak_err_vmap(ubis, origins, compressed_a1["cen"], compressed_a1["hkl"],
                            compressed_a1["wave"], compressed_a1["kvec"], mask_r)
    med = jnp.nanmedian(jnp.where(mask_r, e, jnp.nan), axis=1)

    keep   = mask_r & (e < 4.0 * med[:, None])
    print(f"iter {it}: dropped {int((mask_r & ~keep).sum())} peaks, "
          f"min/grain now {int(keep.sum(axis=1).min())}")
    mask_r = keep

ubis_fit_a1, origins_sample_fit_a1, all_losses_a1 = ubis, origins, losses

In [ ]:
# convergence check

print("mask_r peaks:", int(mask_r.sum()),
      " of", int(compressed_a1["mask"].sum()))

pert = jnp.full_like(all_origins_a1, 1.0 / 1000.0)   # 1 um real (scaled units)

for n in (1, 2, 5, 20):
    _, o_n, _ = refine_vmap(jnp.array(idx_a1.ubis), all_origins_a1,
                            compressed_a1["cen"], compressed_a1["hkl"],
                            compressed_a1["wave"], compressed_a1["kvec"],
                            mask_r, n, 1e-2, 1000.0)
    _, o_p, _ = refine_vmap(jnp.array(idx_a1.ubis), all_origins_a1 + pert,
                            compressed_a1["cen"], compressed_a1["hkl"],
                            compressed_a1["wave"], compressed_a1["kvec"],
                            mask_r, n, 1e-2, 1000.0)
    s = jnp.linalg.norm(o_n - o_p, axis=1)
    print(n, "steps -> shift p50/max:",
          float(jnp.percentile(s, 50)), float(s.max()))

In [ ]:
fig, axs = plt.subplots(2,1,sharex=True,sharey=True)
axs[0].plot(all_losses_a1.T)
axs[1].plot(all_losses_a1.mean(axis=0), label='a1')
axs[0].set(yscale='log')
axs[1].set(yscale='log')
axs[0].set(title='Loss over all grains')
axs[1].set(xlabel='Epoch', ylabel='Loss',title='Mean loss')
plt.show()

### Just $K_{\alpha_{2}}$

In [ ]:
%%time

all_origins_a2 = jnp.zeros((len(idx_a2.ubis), 3))

# Pack the data into grain-specific buffers (num_grains, M, ...)
grain_ids_a2 = jnp.arange(len(idx_a2.ubis))
compressed_a2 = v_pack(grain_ids_a2, ga_a2, centroid_with_error[mask_a2], hkli_a2, wavelengths_a2[mask_a2], kvecs[mask_a2])

mask_r = compressed_a2["mask"]

for it in range(3):
    ubis, origins, losses = refine_vmap(
        jnp.array(idx_a2.ubis), all_origins_a2,
        compressed_a2["cen"],  compressed_a2["hkl"], compressed_a2["wave"],
        compressed_a2["kvec"], mask_r,
        5, 1e-2, 1000.0)

    e   = per_peak_err_vmap(ubis, origins, compressed_a2["cen"], compressed_a2["hkl"],
                            compressed_a2["wave"], compressed_a2["kvec"], mask_r)
    med = jnp.nanmedian(jnp.where(mask_r, e, jnp.nan), axis=1)

    keep   = mask_r & (e < 4.0 * med[:, None])
    print(f"iter {it}: dropped {int((mask_r & ~keep).sum())} peaks, "
          f"min/grain now {int(keep.sum(axis=1).min())}")
    mask_r = keep

ubis_fit_a2, origins_sample_fit_a2, all_losses_a2 = ubis, origins, losses

### Both $K_{\alpha_{1}}$ and $K_{\alpha_{2}}$

In [ ]:
%%time

all_origins_both = jnp.zeros((len(idx_both.ubis), 3))
centroids_both = jnp.concatenate([centroid_with_error[mask_a1], centroid_with_error[mask_a2]])
wavelengths_both = jnp.concatenate([wavelengths_a1[mask_a1], wavelengths_a2[mask_a2]])
kvecs_both = jnp.concatenate([kvecs[mask_a1], kvecs[mask_a2]])

# Pack the data into grain-specific buffers (num_grains, M, ...)
grain_ids_both = jnp.arange(len(idx_both.ubis))
compressed_both = v_pack(grain_ids_both, ga_both, centroids_both, hkli_both, wavelengths_both, kvecs_both)

mask_r = compressed_both["mask"]

for it in range(3):
    ubis, origins, losses = refine_vmap(
        jnp.array(idx_both.ubis), all_origins_both,
        compressed_both["cen"],  compressed_both["hkl"], compressed_both["wave"],
        compressed_both["kvec"], mask_r,
        5, 1e-2, 1000.0)

    e   = per_peak_err_vmap(ubis, origins, compressed_both["cen"], compressed_both["hkl"],
                            compressed_both["wave"], compressed_both["kvec"], mask_r)
    med = jnp.nanmedian(jnp.where(mask_r, e, jnp.nan), axis=1)

    keep   = mask_r & (e < 4.0 * med[:, None])
    print(f"iter {it}: dropped {int((mask_r & ~keep).sum())} peaks, "
          f"min/grain now {int(keep.sum(axis=1).min())}")
    mask_r = keep

ubis_fit_both, origins_sample_fit_both, all_losses_both = ubis, origins, losses

In [ ]:
fig, ax = plt.subplots()
ax.plot(all_losses_a1.mean(axis=0), label='a1')
ax.plot(all_losses_a2.mean(axis=0), label='a2')
ax.plot(all_losses_both.mean(axis=0), label='both')
ax.set(yscale='log')
ax.legend()
ax.set(xlabel='Epoch', ylabel='loss', title='Loss over all grains')
plt.show()

## Match results to ground truth
As the indexer returns UBIs in a different order, we need to match the refinement results to the ground-truth grains.  
First, we make grain lists from the refined results.  
Then, we map each of the grain lists (including the ground truth grains) into the fundamental zone by maximising traces.  
Then we look for matches with a 6D translation-orientation feature vector $k$-d tree. This is highly biased towards orientations because they should more more reliable than translations.

In [ ]:
SYM = jnp.array(ROTATIONS[6])   # hexagonal crystal-frame ops

def cast_to_fundamental_zone(U, symmetry_ops):
    best_U = U
    max_trace = jnp.trace(U)
    
    for S in symmetry_ops:

        U_equiv = jnp.matmul(U, S)
        
        current_trace = jnp.trace(U_equiv)
        
        if current_trace > max_trace:
            max_trace = current_trace
            best_U = U_equiv
            
    return best_U

def move_grains_to_fz(gl, sym_ops):
    newgl = []
    for g in gl:
        best_U = cast_to_fundamental_zone(g.U, sym_ops)
        new_UB = best_U @ g.B
        new_UBI = jnp.linalg.inv(new_UB)
        newg = ImageD11.grain.grain(new_UBI, translation=g.translation)
        newg.ref_unitcell = ref_unitcell
        newgl.append(newg)

    return newgl

def match_grains(grains1, grains2, W=50, dist_tol=100):
    """Returns (distance, index into grains1, index into grains2), all same length."""
    def desc(gl):
        return jnp.column_stack([
            jnp.array([jnp.asarray(g.translation) for g in gl]),
            W * jnp.array([Rotation.from_matrix(jnp.asarray(g.U)).as_rotvec() for g in gl])])
    d, m = scipy.spatial.cKDTree(desc(grains2)).query(desc(grains1), k=1,
                                                      distance_upper_bound=dist_tol)
    valid = jnp.isfinite(d)
    return d[valid], jnp.nonzero(valid)[0], m[valid]

@jax.jit
def gt_ubi_aligned(U_gt, U_fit, B):
    """GT UBI in the same symmetry setting as the fitted grain."""
    M = U_fit.T @ U_gt
    S = SYM[jnp.argmax(jnp.trace(M @ SYM, axis1=1, axis2=2))]
    return jnp.linalg.inv((U_gt @ S) @ B)

@jax.jit
def loss_at_fixed_ubi(ubi, origin, cen, hkl, wave, kvec, mask):
    """Mean |dhkl| for a GIVEN ubi — no analytical UB refit."""
    q = detector_to_q_vec(cen[:, 0], cen[:, 1], cen[:, 2], wave, kvec, origin)
    d = (ubi @ q.T).T - hkl
    e = jnp.sqrt(jnp.sum(d * d, axis=1) + 1e-24)
    mf = mask.astype(e.dtype)
    return jnp.sum(e * mf) / (jnp.sum(mf) + 1e-10) + 1e-10

gt_loss_vmap = jax.vmap(loss_at_fixed_ubi, in_axes=(0, 0, 0, 0, 0, 0, 0))

# ---------- refined grain lists ----------

grains_a1   = [ImageD11.grain.grain(ubis_fit_a1[g],   translation=origins_sample_fit_a1[g])   for g in range(len(idx_a1.ubis))]
grains_a2   = [ImageD11.grain.grain(ubis_fit_a2[g],   translation=origins_sample_fit_a2[g])   for g in range(len(idx_a2.ubis))]
grains_both = [ImageD11.grain.grain(ubis_fit_both[g], translation=origins_sample_fit_both[g]) for g in range(len(idx_both.ubis))]

oriens_a1   = jnp.stack([g.U for g in grains_a1])
oriens_a2   = jnp.stack([g.U for g in grains_a2])
oriens_both = jnp.stack([g.U for g in grains_both])

# ---------- match ----------

grains_fz      = move_grains_to_fz(grains,      ROTATIONS[6])
grains_a1_fz   = move_grains_to_fz(grains_a1,   ROTATIONS[6])
grains_a2_fz   = move_grains_to_fz(grains_a2,   ROTATIONS[6])
grains_both_fz = move_grains_to_fz(grains_both, ROTATIONS[6])

dist_a1,   fit_a1,   gt_a1   = match_grains(grains_a1_fz,   grains_fz)
dist_a2,   fit_a2,   gt_a2   = match_grains(grains_a2_fz,   grains_fz)
dist_both, fit_both, gt_both = match_grains(grains_both_fz, grains_fz)

for tag, fit, gt, n in [('a1', fit_a1, gt_a1, len(grains_a1)),
                        ('a2', fit_a2, gt_a2, len(grains_a2)),
                        ('both', fit_both, gt_both, len(grains_both))]:
    print(f"{tag}: {len(fit)}/{n} matched, {len(gt) - len(jnp.unique(gt))} GT grains claimed twice")

# ---------- ground-truth losses ----------

def gt_losses(gt_idx, fit_idx, oriens, packed):
    ubi_gt = jax.vmap(gt_ubi_aligned, in_axes=(0, 0, None))(
        U_matrices[gt_idx], oriens[fit_idx], struc.B)
    return gt_loss_vmap(ubi_gt, translations_sample[gt_idx],
                        packed["cen"][fit_idx],  packed["hkl"][fit_idx],
                        packed["wave"][fit_idx], packed["kvec"][fit_idx],
                        packed["mask"][fit_idx])

gt_losses_a1   = gt_losses(gt_a1,   fit_a1,   oriens_a1,   compressed_a1)
gt_losses_a2   = gt_losses(gt_a2,   fit_a2,   oriens_a2,   compressed_a2)
gt_losses_both = gt_losses(gt_both, fit_both, oriens_both, compressed_both)

# ---------- metrics ----------

pos_diff_a1   = jnp.linalg.norm(translations_sample[gt_a1]   - origins_sample_fit_a1[fit_a1],     axis=1)
pos_diff_a2   = jnp.linalg.norm(translations_sample[gt_a2]   - origins_sample_fit_a2[fit_a2],     axis=1)
pos_diff_both = jnp.linalg.norm(translations_sample[gt_both] - origins_sample_fit_both[fit_both], axis=1)

misorien_a1   = jnp.array([jnp.min(Umis(U_matrices[g], oriens_a1[f],   6)[:, 1]) for g, f in zip(gt_a1,   fit_a1)])
misorien_a2   = jnp.array([jnp.min(Umis(U_matrices[g], oriens_a2[f],   6)[:, 1]) for g, f in zip(gt_a2,   fit_a2)])
misorien_both = jnp.array([jnp.min(Umis(U_matrices[g], oriens_both[f], 6)[:, 1]) for g, f in zip(gt_both, fit_both)])

strains_a1   = jnp.array([grains_a1[f].eps_sample_matrix(struc.lattice_parameters)   for f in fit_a1])
strains_a2   = jnp.array([grains_a2[f].eps_sample_matrix(struc.lattice_parameters)   for f in fit_a2])
strains_both = jnp.array([grains_both[f].eps_sample_matrix(struc.lattice_parameters) for f in fit_both])

strains_norm_a1   = jnp.sqrt((strains_a1**2).sum(axis=(1,2)))
strains_norm_a2   = jnp.sqrt((strains_a2**2).sum(axis=(1,2)))
strains_norm_both = jnp.sqrt((strains_both**2).sum(axis=(1,2)))

loss_diff_a1   = all_losses_a1[fit_a1, -1]     - gt_losses_a1
loss_diff_a2   = all_losses_a2[fit_a2, -1]     - gt_losses_a2
loss_diff_both = all_losses_both[fit_both, -1] - gt_losses_both

print("refined:", float(all_losses_a1[fit_a1, -1].mean()),
      " ground truth:", float(gt_losses_a1.mean()))

In [ ]:
fig, axs = plt.subplots(3,1,sharex=True,sharey=True,constrained_layout=True)
axs[0].hist(pos_diff_a1, bins=20)
axs[1].hist(pos_diff_a2, bins=20)
axs[2].hist(pos_diff_both, bins=20)
axs[0].set_title(r'Just $K_{\alpha_1}$')
axs[1].set_title(r'Just $K_{\alpha_2}$')
axs[2].set_title(r'$K_{\alpha_1}$ and $K_{\alpha_2}$')
fig.supxlabel(r'Position error to ground truth (μm)')
fig.supylabel('Counts')
plt.show()

In [ ]:
fig, axs = plt.subplots(3,1,sharex=True,sharey=True,constrained_layout=True)
axs[0].hist(strains_norm_a1*1e3, bins=20)
axs[1].hist(strains_norm_a2*1e3, bins=20)
axs[2].hist(strains_norm_both*1e3, bins=20)
axs[0].set_title(r'Just $K_{\alpha_1}$')
axs[1].set_title(r'Just $K_{\alpha_2}$')
axs[2].set_title(r'$K_{\alpha_1}$ and $K_{\alpha_2}$')
fig.supxlabel(r'Strain norm (should be 0) (x1e-3)')
fig.supylabel('Counts')
plt.show()

In [ ]:
fig, axs = plt.subplots(3,1,sharex=True,sharey=True,constrained_layout=True)
axs[0].hist(misorien_a1, bins=20)
axs[1].hist(misorien_a2, bins=20)
axs[2].hist(misorien_both, bins=20)
axs[0].set_title(r'Just $K_{\alpha_1}$')
axs[1].set_title(r'Just $K_{\alpha_2}$')
axs[2].set_title(r'$K_{\alpha_1}$ and $K_{\alpha_2}$')
fig.supxlabel('Misorientation to ground truth (deg)')
fig.supylabel('Counts')
plt.show()

In [ ]:
fig, axs = plt.subplots(3,1,sharex=True,sharey=True,constrained_layout=True)
axs[0].hist(all_losses_a1[fit_a1, -1], bins=20, alpha=0.5, label='refined')
axs[0].hist(gt_losses_a1, bins=20, alpha=0.5, label='ground truth')
axs[1].hist(all_losses_a2[fit_a2, -1], bins=20, alpha=0.5)
axs[1].hist(gt_losses_a2, bins=20,  alpha=0.5)
axs[2].hist(all_losses_both[fit_both, -1], bins=20,alpha=0.5)
axs[2].hist(gt_losses_both, bins=20, alpha=0.5)
axs[0].legend()
axs[0].set_title(r'Just $K_{\alpha_1}$')
axs[1].set_title(r'Just $K_{\alpha_2}$')
axs[2].set_title(r'$K_{\alpha_1}$ and $K_{\alpha_2}$')
fig.supxlabel('Final loss function')
fig.supylabel('Counts')
plt.show()

In [ ]:
fig, axs = plt.subplots(3,1,sharex=True,sharey=True,constrained_layout=True)
axs[0].hist(loss_diff_a1, bins=20)
axs[1].hist(loss_diff_a2, bins=20)
axs[2].hist(loss_diff_both, bins=20)
axs[0].set_title(r'Just $K_{\alpha_1}$')
axs[1].set_title(r'Just $K_{\alpha_2}$')
axs[2].set_title(r'$K_{\alpha_1}$ and $K_{\alpha_2}$')
fig.supxlabel('Difference in loss function to ground truth')
fig.supylabel('Counts')
plt.show()

## Statistics for paper

In [ ]:
def print_grain_metrics(
    pos_diff_a1, pos_diff_a2, pos_diff_both,
    misorien_a1, misorien_a2, misorien_both,
    strains_norm_a1, strains_norm_a2, strains_norm_both,
    loss_diff_a1, loss_diff_a2, loss_diff_both
):
    # Defining metrics with their specific scaling factors
    # (Label, data1, data2, databoth, scale_factor, unit_label)
    metrics = [
        ("Loss Diff", loss_diff_a1, loss_diff_a2, loss_diff_both, 1e6, "(x1e-6)"),
        ("Positional Diff", pos_diff_a1, pos_diff_a2, pos_diff_both, 1.0, ""),
        ("Misorientation", misorien_a1, misorien_a2, misorien_both, 1e3, "(x1e-3)"),
        ("Strain Norm", strains_norm_a1, strains_norm_a2, strains_norm_both, 1e4, "(x1e-4)"),
    ]

    header = f"{'Metric':<25} | {'a1 Mean':<12} | {'a2 Mean':<12} | {'Both Mean':<12} | {'Comb. Frac':<12}"
    print(header)
    print("-" * len(header))

    for label, d1, d2, db, scale, unit in metrics:
        # Apply scaling and calculate means
        m1 = jnp.mean(d1).item() * scale
        m2 = jnp.mean(d2).item() * scale
        mb = jnp.mean(db).item() * scale
        
        m_avg_indiv = (m1 + m2) / 2
        
        # The fraction remains the same regardless of scaling
        fraction = mb / m_avg_indiv if m_avg_indiv != 0 else 0
        
        full_label = f"{label} {unit}".strip()
        print(f"{full_label:<25} | {m1:>12.6f} | {m2:>12.6f} | {mb:>12.6f} | {fraction:>12.4f}")

print_grain_metrics(
    pos_diff_a1, pos_diff_a2, pos_diff_both,
    misorien_a1, misorien_a2, misorien_both,
    strains_norm_a1, strains_norm_a2, strains_norm_both,
    loss_diff_a1, loss_diff_a2, loss_diff_both
)

In [ ]:
# total peaks (a1 and a2)
mask_a1.sum(), mask_a1.sum()*2, mask_a1.sum()/n_grains, mask_a1.sum()*2/n_grains

In [ ]:
p = jnp.asarray(pos_diff_a1)
print(f"median {jnp.median(p):.2f}  p90 {jnp.percentile(p,90):.2f}  "
      f"p99 {jnp.percentile(p,99):.2f}  max {p.max():.2f}")

## Plots for paper

In [ ]:
fig, ax = plt.subplots(figsize=(3.54, 3.54), layout='constrained')
ax.scatter(centroid_with_error[:mask_a1.sum()][:, 1], centroid_with_error[:mask_a1.sum()][:, 0], label=r'$K_{\alpha_{1}}$ peaks', s=.5)
ax.scatter(centroid_with_error[mask_a1.sum():][:, 1], centroid_with_error[mask_a1.sum():][:, 0], label=r'$K_{\alpha_{2}}$ peaks', s=.5)
ax.set_aspect(1)
ax.set(title='Dual-wavelength')
ax.set_xticks([])
ax.set_yticks([])
ax.legend(loc='upper right')
plt.show()
plt.savefig('dual_peaks.png', dpi=600)

In [ ]:
end = time.time()
print(f'Took {end-start:.0f} seconds')